In [ ]:
!pip install faiss-cpu
!pip install duckdb
import numpy as np
import faiss
import torch
from transformers import AutoProcessor, AutoModel
import pickle
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import duckdb
import pandas as pd
import csv
import json
from typing import Any, Optional

# Tính Cosine Similarity 

In [ ]:
model_name = "google/siglip2-so400m-patch16-naflex" 
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

query_text = "nhảy lân và đánh trống"

# Tokenize text
inputs = processor(text=[query_text], padding="max_length", return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    text_features = model.get_text_features(**inputs)

query_vector = text_features.pooler_output.cpu().numpy()

# Định dạng và chuẩn hóa Vector cho FAISS
# Đảm bảo mảng float32 và contiguous
query_vector = np.ascontiguousarray(query_vector.astype(np.float32))

# BẮT BUỘC CHUẨN HÓA L2 ĐỂ TÍNH COSINE SIMILARITY
faiss.normalize_L2(query_vector)

index_path = '/kaggle/input/datasets/tienduongtruong/aic26-b2-data/siglip_features.index'
index = faiss.read_index(index_path)

k = 200
distances, indices = index.search(query_vector, k)
distances, indices = distances[0], indices[0]

with open('/kaggle/input/datasets/tienduongtruong/aic26-b2-data/vectormapping.pkl', 'rb') as f:
    image_paths = pickle.load(f)

results = []
for rank, (faiss_idx, score) in enumerate(zip(indices, distances)):
    mapping_str = image_paths[faiss_idx]
    parts = mapping_str.split('/')
    
    if len(parts) == 2:
        video_name = parts[0]        
        n_keyframes = int(parts[1])
        
    results.append({
        'n_keyframes': n_keyframes,
        'video_name': video_name,
        'cosine_score': float(score),
        'rank': rank + 1
    })

top_k_df = pd.DataFrame(results)
top_k_df

# Lôi đầu keyframes dậy check

In [ ]:
def show_keyframe(video_name, frame_idx):
    part_name = video_name.split('_')[0] # Lấy chữ 'L24'

    if part_name in ['L21', 'L22', 'L23', 'L24', 'L25']:
        base_dir = f'/kaggle/input/datasets/tienduongtruong/aic26-b2-l21-25/Keyframes_{part_name}/Keyframes_{part_name}/{video_name}'
    elif part_name in ['L27', 'L28', 'L29', 'L30']:
        base_dir = f'/kaggle/input/datasets/tienduongtruong/aic26-b2-l27-30/Keyframes_{part_name}/Keyframes_{part_name}/{video_name}'
    elif part_name == 'L26':
        base_dir = f'/kaggle/input/datasets/tienduongtruong/aic26-b2-l26/Keyframes_*/Keyframes_*/{video_name}'
    else:
        print("Nhập tên đúng định dạng đi cha")
        return
    possible_dirs = glob.glob(base_dir) if '*' in base_dir else [base_dir]
    for d in possible_dirs:
        if not os.path.exists(d): 
            continue
            
    padded_idx = str(frame_idx).zfill(4)
    file_name = f"{padded_idx}.jpg"
    
    for d in possible_dirs:
        temp_path = os.path.join(d, file_name)
        if os.path.exists(temp_path):
            image_path = temp_path
            break

    if image_path and os.path.exists(image_path):
        img = Image.open(image_path)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Video: {video_name} | Frame: {file_name}")
        plt.show()
    else:
        print(f"❌ Không tìm thấy file ảnh gốc: {file_name} trong video {video_name}")

In [ ]:
for i in range(90, 100):
    show_keyframe("L30_V031", i)

# Đọc file csv mapkeyframes

In [ ]:
def read_csv(file_path):
    data = []
    with open(file_path, mode='r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)
        for row in csv_reader:
            data.append(row)
    print(f"Đã đọc thành công {len(data)} dòng từ file.")
    return data

In [ ]:
data = read_csv('/kaggle/input/datasets/tienduongtruong/aic26-b2-data/mapkeyframes/mapkeyframes/L30_V031.csv')
data

# Đọc file json

In [ ]:
def read_json(file_path: str) -> Optional[Any]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ Đã đọc thành công file JSON từ {file_path}")
    return data

In [ ]:
data = read_json('/kaggle/input/datasets/tienduongtruong/aic26-b2-data/transcripts/transcripts/L21_V001.json')
data

# Định dạng k Semantic frames từ k keyframes

In [ ]:
# Chạy cosine similarity ở phía trên để lấy top k keyframe trước

In [ ]:
# Hàm đọc các file dữ liệu
def load_metadata_df(json_path): # batch sau sẽ định dạng lại để đồng bộ với file json của object detection
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    video_id = data['video_id']
    rows = []
    for frame in data['frames']:
        # '0063.jpg' -> 63
        frame_idx = int(frame['frame_id'].replace('.jpg', '')) 
        rows.append({
            'video_id': video_id,
            'frame_idx': frame_idx,
            'caption': frame.get('caption', ''),
            'ocr_text': frame.get('ocr_text', '').replace('\n', ' ')
        })
    return pd.DataFrame(rows)

def load_object_detection_df(json_path):
    """Đọc Object JSON, n tương ứng với frame_idx"""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    rows = []
    for item in data:
        rows.append({
            'video_id': video_id,
            'frame_idx': item['n'],
            'objects_dict': json.dumps(item['objects'], ensure_ascii=False) 
        })
    return pd.DataFrame(rows)

def load_transcript_df(json_path):
    video_id = os.path.basename(json_path).split('.')[0]
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    df['video_id'] = video_id
    
    return df

In [ ]:
unique_videos = top_k_df['video_name'].unique()
df_meta_list = []
df_obj_list = []
df_map_list = []
df_trans_list = []

base_dir = '/kaggle/input/datasets/tienduongtruong/aic26-b2-data'

for vid in unique_videos:
    meta_path = f"{base_dir}/metadata/metadata/{vid}.json"
    if os.path.exists(meta_path):
        df_meta_list.append(load_metadata_df(meta_path))

    obj_path = f"{base_dir}/object_detect/{vid}.json"
    if os.path.exists(obj_path):
        df_obj_list.append(load_object_detection_df(obj_path, vid))

    map_path = f"{base_dir}/mapkeyframes/mapkeyframes/{vid}.csv"
    if os.path.exists(map_path):
        df_map = pd.read_csv(map_path)
        df_map['video_id'] = vid
        df_map_list.append(df_map)
        
    trans_path = f"{base_dir}/transcripts/{vid}.json" # Đổi đuôi file thành .json
    if os.path.exists(trans_path):
        df_trans_list.append(load_transcript_df(trans_path, vid))

df_meta = pd.concat(df_meta_list, ignore_index=True) if df_meta_list else pd.DataFrame()
df_obj = pd.concat(df_obj_list, ignore_index=True) if df_obj_list else pd.DataFrame()
df_map = pd.concat(df_map_list, ignore_index=True) if df_map_list else pd.DataFrame()
df_trans = pd.concat(df_trans_list, ignore_index=True) if df_trans_list else pd.DataFrame()

In [ ]:
query = """
SELECT 
    k.rank,
    k.video_name,
    k.n_keyframes,
    k.cosine_score,
    m.pts_time,
    meta.caption,
    meta.ocr_text,
    obj.objects_dict,
    t.text AS transcript_text
FROM top_k_df AS k

-- Join 1: Ánh xạ để lấy thời gian pts_time của frame hiện tại
LEFT JOIN df_map AS m 
    ON k.video_name = m.video_id AND k.n_keyframes = m.frame_idx

-- Join 2: Lấy metadata (caption, ocr) 
LEFT JOIN df_meta AS meta 
    ON k.video_name = meta.video_id AND k.n_keyframes = meta.frame_idx

-- Join 3: Lấy object detection
LEFT JOIN df_obj AS obj 
    ON k.video_name = obj.video_id AND k.n_keyframes = obj.frame_idx

-- Join 4: Bắt câu thoại (Transcript) kẹp giữa thời gian của frame
LEFT JOIN df_trans AS t
    ON k.video_name = t.video_id 
    AND m.pts_time >= t.start_pts 
    AND m.pts_time <= t.end_pts

ORDER BY k.rank ASC
"""

# Thực thi Query
semantic_frames_df = duckdb.query(query).df()

In [ ]:
sematic_frames_df